# Data Cleaning: FIFA 21 Player Ratings Dataset

**Objective:** Take a deliberately messy real-world dataset and systematically transform it into a clean, analysis-ready dataset — documenting every decision along the way.

**Dataset:** `fifa21_raw_data.csv` — 18,979 FIFA 21 player records, 77 columns, scraped directly from a player ratings site. Many columns contain formatting artifacts typical of scraped/raw data: currency symbols, unit suffixes, star-rating symbols, and stray whitespace/newline characters.

**Tools:** Python, pandas, numpy, Jupyter Notebook


## 1. Setup & Imports

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

## 2. Load Dataset & Data Quality Report

In [2]:
df = pd.read_csv('fifa21_raw_data.csv', low_memory=False)

print("Shape:", df.shape)
df.head(3)

Shape: (18979, 77)


,photoUrl,LongName,playerUrl,Nationality,Positions,Name,Age,↓OVA,POT,Team & Contract,ID,Height,Weight,foot,BOV,BP,Growth,Joined,Loan Date End,Value,Wage,Release Clause,Attacking,Crossing,Finishing,Heading Accuracy,Short Passing,Volleys,Skill,Dribbling,Curve,FK Accuracy,Long Passing,Ball Control,Movement,Acceleration,Sprint Speed,Agility,Reactions,Balance,Power,Shot Power,Jumping,Stamina,Strength,Long Shots,Mentality,Aggression,Interceptions,Positioning,Vision,Penalties,Composure,Defending,Marking,Standing Tackle,Sliding Tackle,Goalkeeping,GK Diving,GK Handling,GK Kicking,GK Positioning,GK Reflexes,Total Stats,Base Stats,W/F,SM,A/W,D/W,IR,PAC,SHO,PAS,DRI,DEF,PHY,Hits
0,https://cdn.sofifa.com/players/158/023/21_60.png,Lionel Messi,http://sofifa.com/player/158023/lionel-messi/2...,Argentina,RW ST CF,L. Messi,33,93,93,\n\n\n\nFC Barcelona\n2004 ~ 2021\n\n,158023,"5'7""",159lbs,Left,93,RW,0,"Jul 1, 2004",NaN,€67.5M,€560K,€138.4M,429,85,95,70,91,88,470,96,93,94,91,96,451,91,80,91,94,95,389,86,68,72,69,94,347,44,40,93,95,75,96,91,32,35,24,54,6,11,15,14,8,2231,466,4 ★,4★,Medium,Low,5 ★,85,92,91,95,38,65,\n372
1,https://cdn.sofifa.com/players/020/801/21_60.png,C. Ronaldo dos Santos Aveiro,http://sofifa.com/player/20801/c-ronaldo-dos-s...,Portugal,ST LW,Cristiano Ronaldo,35,92,92,\n\n\n\nJuventus\n2018 ~ 2022\n\n,20801,"6'2""",183lbs,Right,92,ST,0,"Jul 10, 2018",NaN,€46M,€220K,€75.9M,437,84,95,90,82,86,414,88,81,76,77,92,431,87,91,87,95,71,444,94,95,84,78,93,353,63,29,95,82,84,95,84,28,32,24,58,7,11,15,14,11,2221,464,4 ★,5★,High,Low,5 ★,89,93,81,89,35,77,\n344
2,https://cdn.sofifa.com/players/200/389/21_60.png,Jan Oblak,http://sofifa.com/player/200389/jan-oblak/210005/,Slovenia,GK,J. Oblak,27,91,93,\n\n\n\nAtlético Madrid\n2014 ~ 2023\n\n,200389,"6'2""",192lbs,Right,91,GK,2,"Jul 16, 2014",NaN,€75M,€125K,€159.4M,95,13,11,15,43,13,109,12,13,14,40,30,307,43,60,67,88,49,268,59,78,41,78,12,140,34,19,11,65,11,68,57,27,12,18,437,87,92,78,90,90,1413,489,3 ★,1★,Medium,Medium,3 ★,87,92,78,90,52,90,\n86


In [3]:
# --- Data Quality Report ---
nulls = df.isnull().sum()
nulls_present = nulls[nulls > 0]

duplicate_rows = df.duplicated().sum()

print("Columns with missing values:")
print(nulls_present)
print("\nTotal duplicate rows:", duplicate_rows)
print("\nData types:")
print(df.dtypes.value_counts())

Columns with missing values:
Loan Date End    17966
dtype: int64

Total duplicate rows: 1

Data types:
int64    55
str      22
Name: count, dtype: int64


**Observation:** Only `Loan Date End` has missing values (17,966 out of 18,979 rows) — this is expected, since most players are not currently on loan, rather than data being genuinely missing. There is 1 duplicate row. Nearly all columns are stored as `object` (string) type even though many represent numbers (Height, Weight, Value, Wage, star ratings, etc.) — this is the core cleaning challenge in this dataset.

## 3. Handling Missing Values

In [4]:
# 'Loan Date End' being null does not mean missing data — it means the player
# is not on loan. Rather than imputing a fake date, we convert this into an
# informative boolean flag and keep the original column for players who ARE on loan.
df['On Loan'] = df['Loan Date End'].notnull()

print(df['On Loan'].value_counts())

On Loan
False    17966
True      1013
Name: count, dtype: int64


**Justification:** Imputing a date for `Loan Date End` would be meaningless and misleading — there's no "typical" loan end date to fill in. Converting it to a boolean `On Loan` flag preserves the real information (whether a player is on loan) without inventing data. The original column is kept as-is for the small number of players who do have a loan end date.

## 4. Duplicate Removal

In [5]:
rows_before = len(df)
df = df.drop_duplicates()
rows_after = len(df)

print(f"Removed {rows_before - rows_after} duplicate row(s).")
print(f"Rows before: {rows_before}, rows after: {rows_after}")

Removed 1 duplicate row(s).
Rows before: 18979, rows after: 18978


## 5. Standardising Messy Columns

Several columns were scraped as raw text with units, currency symbols, and star symbols mixed into the values. Each is converted below into a clean numeric or categorical form, with the transformation explained.

### 5.1 Height and Weight → numeric (cm / kg)

In [6]:
# Height is stored like 5'7" (feet'inches"). Convert to centimetres.
def height_to_cm(h):
    h = str(h).replace('"', '').strip()
    feet, inches = h.split("'")
    total_inches = int(feet) * 12 + int(inches)
    return round(total_inches * 2.54, 1)

df['Height_cm'] = df['Height'].apply(height_to_cm)

# Weight is stored like 159lbs. Convert to kilograms.
df['Weight_kg'] = df['Weight'].str.replace('lbs', '', regex=False).astype(int) * 0.453592
df['Weight_kg'] = df['Weight_kg'].round(1)

df[['Height', 'Height_cm', 'Weight', 'Weight_kg']].head()

,Height,Height_cm,Weight,Weight_kg
0,"5'7""",170.2,159lbs,72.1
1,"6'2""",188.0,183lbs,83.0
2,"6'2""",188.0,192lbs,87.1
3,"5'11""",180.3,154lbs,69.9
4,"5'9""",175.3,150lbs,68.0


**Justification:** Height and weight are fundamentally numeric measurements, but were stored as text with embedded units (feet/inches, pounds). Converting to a single standard unit (cm, kg) makes them usable for any numeric analysis or modelling. The original columns are kept for traceability; the new `_cm`/`_kg` columns are what should be used going forward.

### 5.2 Monetary columns (Value, Wage, Release Clause) → numeric euros

In [7]:
def money_to_number(x):
    if pd.isnull(x):
        return np.nan
    x = str(x).replace('€', '').strip()
    if x.endswith('M'):
        return float(x[:-1]) * 1_000_000
    elif x.endswith('K'):
        return float(x[:-1]) * 1_000
    elif x in ('', '0'):
        return 0.0
    else:
        return float(x)

for col in ['Value', 'Wage', 'Release Clause']:
    df[col + '_EUR'] = df[col].apply(money_to_number)

df[['Value', 'Value_EUR', 'Wage', 'Wage_EUR', 'Release Clause', 'Release Clause_EUR']].head()

,Value,Value_EUR,Wage,Wage_EUR,Release Clause,Release Clause_EUR
0,€67.5M,67500000.0,€560K,560000.0,€138.4M,138400000.0
1,€46M,46000000.0,€220K,220000.0,€75.9M,75900000.0
2,€75M,75000000.0,€125K,125000.0,€159.4M,159400000.0
3,€87M,87000000.0,€370K,370000.0,€161M,161000000.0
4,€90M,90000000.0,€270K,270000.0,€166.5M,166500000.0


**Justification:** These columns mix `€`, `M` (million), and `K` (thousand) shorthand into a single string, making them unusable for any calculation or sorting as-is. Parsing them into a plain numeric euro value (`_EUR` suffix) allows direct comparison, aggregation, and use in models.

### 5.3 Star-rating columns (W/F, SM, IR) → integer scale

In [8]:
# These columns store a 1-5 star rating as text, with inconsistent spacing
# around the star symbol (e.g. '4 ★' vs '4★').
for col in ['W/F', 'SM', 'IR']:
    df[col + '_num'] = df[col].str.extract(r'(\d+)').astype(int)

df[['W/F', 'W/F_num', 'SM', 'SM_num', 'IR', 'IR_num']].head()

,W/F,W/F_num,SM,SM_num,IR,IR_num
0,4 ★,4,4★,4,5 ★,5
1,4 ★,4,5★,5,5 ★,5
2,3 ★,3,1★,1,3 ★,3
3,5 ★,5,4★,4,4 ★,4
4,5 ★,5,5★,5,5 ★,5


**Justification:** The star symbol carries no analytical value once we know the numeric rating — extracting just the digit gives a clean integer scale (1-5) that can be used directly in statistics or visualisations.

### 5.4 'Hits' column → clean integer

In [9]:
# Hits contains a stray newline character before each number (scraping artifact),
# and some values use a 'K' suffix for thousands (e.g. '1.5K' = 1500), the same
# shorthand seen in the monetary columns.
print("Before:", df['Hits'].unique()[:5])

def hits_to_number(x):
    if pd.isnull(x):
        return np.nan
    x = str(x).replace('\n', '').strip()
    if x.endswith('K'):
        return float(x[:-1]) * 1_000
    return float(x)

df['Hits'] = df['Hits'].apply(hits_to_number)

print("After:", df['Hits'].head())
print("Nulls after conversion:", df['Hits'].isnull().sum())

Before: <StringArray>
['\n372', '\n344', '\n86', '\n163', '\n273']
Length: 5, dtype: str
After: 0    372.0
1    344.0
2     86.0
3    163.0
4    273.0
Name: Hits, dtype: float64
Nulls after conversion: 0


**Justification:** The leading `\n` character is purely a scraping artifact from the source website and carries no meaning. A first pass revealed 15 values used a `K` suffix (e.g. `'1.5K'` for popular young players like Haaland and Bellingham) — the same shorthand as the monetary columns — which a naive numeric conversion would silently turn into missing values. Handling the `K` suffix explicitly avoids losing real data for these rows.

### 5.5 'Team & Contract' → separate Club and Contract Status columns

In [10]:
# This column bundles the club (or nationality, for free agents) and a contract
# status together, wrapped in multiple leading/trailing newlines from scraping,
# e.g. '\n\n\n\nFC Barcelona\n2004 ~ 2021\n\n'. A quick check shows every single
# row splits into exactly 2 non-empty lines once blank lines are dropped — but
# the second line takes 3 different forms depending on the player's situation:
#   - a year range for a contracted player, e.g. '2004 ~ 2021'
#   - a loan end date for a player currently on loan, e.g. 'Jun 30, 2021 On Loan'
#   - 'Free' for a free agent (whose first line is their nationality, not a club)
def split_team_contract(x):
    parts = [p.strip() for p in str(x).split('\n') if p.strip() != '']
    if len(parts) == 2:
        return pd.Series(parts)
    return pd.Series([np.nan, np.nan])

df[['Club', 'Contract Status']] = df['Team & Contract'].apply(split_team_contract)

df[['Team & Contract', 'Club', 'Contract Status']].head()

,Team & Contract,Club,Contract Status
0,\n\n\n\nFC Barcelona\n2004 ~ 2021\n\n,FC Barcelona,2004 ~ 2021
1,\n\n\n\nJuventus\n2018 ~ 2022\n\n,Juventus,2018 ~ 2022
2,\n\n\n\nAtlético Madrid\n2014 ~ 2023\n\n,Atlético Madrid,2014 ~ 2023
3,\n\n\n\nManchester City\n2015 ~ 2023\n\n,Manchester City,2015 ~ 2023
4,\n\n\n\nParis Saint-Germain\n2017 ~ 2022\n\n,Paris Saint-Germain,2017 ~ 2022


In [11]:
# Confirm the split worked for every row (no unexpected nulls introduced)
print("Rows where split failed:", df['Club'].isnull().sum())

# Sanity check across the 3 known patterns
print(df[df['Contract Status'].str.contains('On Loan', na=False)][['Name','Club','Contract Status']].head(2))
print(df[df['Contract Status'] == 'Free'][['Name','Club','Contract Status']].head(2))

Rows where split failed: 0
               Name                 Club       Contract Status
205         G. Bale    Tottenham Hotspur  Jun 30, 2021 On Loan
250  Danilo Pereira  Paris Saint-Germain  Jun 30, 2021 On Loan
               Name    Club Contract Status
288  Welington Dano  Brazil            Free
292  Juiano Mestres  Brazil            Free


**Justification:** An initial regex-based approach (looking specifically for a `YYYY ~ YYYY` year range) silently failed on club names containing non-ASCII characters, and on the two other real patterns in this column — players on loan (`'Jun 30, 2021 On Loan'`) and free agents (`'Free'`, with nationality instead of a club on the first line). Splitting structurally on non-empty lines instead of matching a specific text pattern is more reliable and correctly captures all three cases, which was confirmed by checking that zero rows failed to split and by spot-checking each pattern above.

### 5.6 Categorical text — check for inconsistent casing/spelling

In [12]:
for col in ['foot', 'A/W', 'D/W']:
    print(col, '->', sorted(df[col].dropna().unique()))

foot -> ['Left', 'Right']
A/W -> ['High', 'Low', 'Medium']
D/W -> ['High', 'Low', 'Medium']


**Observation:** `foot`, `A/W` (Attacking Work Rate), and `D/W` (Defensive Work Rate) are already consistently formatted (`Left`/`Right`, `Low`/`Medium`/`High`) with no casing inconsistencies — no standardisation needed here. This is worth explicitly checking rather than assuming, since it confirms the data doesn't need further cleaning at this step.

## 6. Outlier Detection (IQR Method)

In [13]:
def iqr_outliers(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return series[(series < lower) | (series > upper)], lower, upper

for col in ['Age', 'Value_EUR', 'Wage_EUR', '↓OVA']:
    outliers, lower, upper = iqr_outliers(df[col])
    print(f"{col}: {len(outliers)} outliers  (bounds: {lower:.1f} to {upper:.1f})")

Age: 8 outliers  (bounds: 9.0 to 41.0)
Value_EUR: 2662 outliers  (bounds: -1950000.0 to 4050000.0)
Wage_EUR: 2277 outliers  (bounds: -9500.0 to 18500.0)
↓OVA: 156 outliers  (bounds: 47.5 to 83.5)


**Decision per column:**
- **Age:** Outliers here are simply very young or very old professional players — a legitimate, real part of the data. **Retained**, no action taken.
- **Value_EUR / Wage_EUR:** High-value outliers are elite players (e.g. Messi, Ronaldo) with genuinely exceptional market value — not data errors. **Retained**, since removing them would delete meaningful information about star players.
- **↓OVA (Overall rating):** Bound by the game's own 1-99 rating scale, so extreme values are expected and valid. **Retained.**

In this dataset, IQR-flagged "outliers" mostly represent genuinely exceptional players rather than data entry errors, so the appropriate action is to retain them rather than cap or remove — this is itself a documented decision, not an oversight.

## 7. Final Data Type Correction

In [14]:
# Ensure key columns have the correct final dtype
df['ID'] = df['ID'].astype(str)
df['Age'] = df['Age'].astype(int)
df['↓OVA'] = df['↓OVA'].astype(int)
df['Height_cm'] = df['Height_cm'].astype(float)
df['Weight_kg'] = df['Weight_kg'].astype(float)

print(df[['ID', 'Age', '↓OVA', 'Height_cm', 'Weight_kg', 'Value_EUR', 'Hits']].dtypes)

ID               str
Age            int64
↓OVA           int64
Height_cm    float64
Weight_kg    float64
Value_EUR    float64
Hits         float64
dtype: object


**Justification:** `ID` is switched to string since it's an identifier, not a value meant for arithmetic. `Age` and `Overall Rating` are confirmed as integers, and the newly engineered numeric columns (`Height_cm`, `Weight_kg`, `Value_EUR`, etc.) are confirmed as float/int as appropriate.

## 8. Before vs. After Summary

In [15]:
summary = pd.DataFrame({
    'Metric': [
        'Total rows',
        'Duplicate rows',
        'Columns with unresolved missing values',
        'Height/Weight stored as text',
        'Monetary columns stored as text',
        'Star ratings stored as text',
        "'Hits' column had formatting artifacts",
    ],
    'Before Cleaning': [rows_before, 1, 1, 'Yes', 'Yes', 'Yes', 'Yes'],
    'After Cleaning':  [len(df), 0, 0, 'No (Height_cm/Weight_kg added)',
                        'No (_EUR columns added)', 'No (_num columns added)', 'No'],
})
summary

,Metric,Before Cleaning,After Cleaning
0,Total rows,18979,18978
1,Duplicate rows,1,0
2,Columns with unresolved missing values,1,0
3,Height/Weight stored as text,Yes,No (Height_cm/Weight_kg added)
4,Monetary columns stored as text,Yes,No (_EUR columns added)
5,Star ratings stored as text,Yes,No (_num columns added)
6,'Hits' column had formatting artifacts,Yes,No


## 9. Save Cleaned Dataset

In [16]:
df.to_csv('fifa21_cleaned.csv', index=False)
print("Cleaned dataset saved as 'fifa21_cleaned.csv' with shape:", df.shape)

Cleaned dataset saved as 'fifa21_cleaned.csv' with shape: (18978, 88)


## 10. Conclusion

The raw FIFA 21 dataset arrived with numeric information trapped inside formatted text — currency symbols, unit suffixes, star symbols, and scraping artifacts like stray newlines. After cleaning:

- 1 duplicate row removed
- `Loan Date End` nulls resolved via a proper `On Loan` boolean flag instead of fabricated imputation
- Height, Weight, Value, Wage, Release Clause, and star-rating columns converted from text into clean numeric fields, with originals retained for traceability
- `Team & Contract` split into separate `Club` and `Contract Years` fields
- `Hits` stripped of formatting artifacts and converted to numeric
- Outliers in Age/Value/Wage/Overall Rating reviewed and deliberately retained, since they represent real elite players rather than data errors

The dataset is now ready for downstream analysis or modelling (e.g. predicting player value from attributes) without any further text-parsing required.
